> ⚠️ **机器翻译提示**：本 notebook 由机器翻译生成，未经人工校对，如发现翻译错误敬请谅解（代码与计算结果和越南语原版完全一致，只有文字说明是机器翻译）。

# 第06单元 — 单因素方差分析、Tukey事后检验、双因素方差分析
第06单元专属 notebook —— 重现单因素方差分析 + Tukey HSD（PSPP可以完成，见 `06_anova.sps`），并额外提供双因素方差分析 + 交互效应（PSPP 2.1.1 不支持 `UNIANOVA`，只能在此运行，使用 `statsmodels`）。

SPSS 对应操作：`Analyze > Compare Means > One-Way ANOVA`（+ Post Hoc Tukey）、`Analyze > General Linear Model > Univariate`（双因素）。

In [ ]:
# 先运行这一格（Colab：点击 ▶）。无需安装任何库。
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """模拟数据集"240人班级" —— 第03-08单元统一使用 seed=2026，
    与仓库中的 lop_hoc_240.csv/.sav 完全一致（非真实数据，但100%可复现）。"""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["男", "女"], n)
    method = rng.choice(["传统教学", "项目式教学"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "项目式教学")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "项目式教学") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("已生成 df：", df.shape)

## 1. 单因素方差分析：数学成绩按3所学校分组

🎯 **这个方法用来回答什么问题？** 单因素方差分析检验：3个或以上组的平均值中，是否至少有一对存在统计显著差异——原假设H0：所有组的总体平均值都相同。重要提示：显著的方差分析结果只说明「某处存在差异」，并不能说明是哪一对存在差异——这是下面第2节Tukey事后检验要回答的问题。

In [ ]:
groups = [g["math"].values for _, g in df.groupby("school")]
f_stat, p_val = stats.f_oneway(*groups)
lev_stat, lev_p = stats.levene(*groups)
print(f"Levene检验：F={lev_stat:.3f}, p={lev_p:.3f}")
for name, g in df.groupby("school"):
    print(f"  {name}校：n={len(g)}, M={g['math'].mean():.2f}")
print(f"方差分析：F(2,{len(df)-3})={f_stat:.3f}, p={p_val:.4f}")

#### 📤 实际输出
A校（n=80）M≈50.67；B校（n=99）M≈47.26；C校（n=61）M≈45.34。F(2,237)=5.87, p=0.003 → 具有显著差异。Levene检验 p=0.292（≥0.05，方差齐性，可以使用Tukey检验）。

## 2. Tukey HSD事后检验 —— 找出究竟是哪一对存在差异

🎯 **这个方法用来回答什么问题？** Tukey HSD 对每一对组分别进行检验，并作出校正，以避免因同时比较多对组而「膨胀」第一类错误率（多重比较问题——比较的组对越多，越容易偶然发现某一对「显著」，而实际上并非如此）。这正好回答了第1节方差分析留下的问题：差异究竟出现在哪里。

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
tukey = pairwise_tukeyhsd(df["math"], df["school"], alpha=0.05)
print(tukey)

#### 📤 实际输出
A-B差值3.41（p≈0.046，显著）；A-C差值5.34（p≈0.003，显著）；B-C差值1.93（p≈0.427，不显著）。同质子集：{A}单独一组，{B, C}为一组。

## 3. 双因素方差分析：数学成绩 ~ 学校 × 性别（含交互效应）
⚠️ PSPP 2.1.1 不支持 `UNIANOVA` 命令（已验证，报错「UNIANOVA is not yet implemented」）—— 此部分只能在 notebook 中运行，使用 `statsmodels.formula.api.ols` + `anova_lm`。

🎯 **这个方法用来回答什么问题？** 双因素方差分析同时检验3个问题：(1) 变量A（学校）是否有主效应，(2) 变量B（性别）是否有主效应，(3) A的影响是否取决于B的水平（即交互效应）。第3个问题无法通过分别运行两次单因素方差分析来回答——这正是需要双因素方差分析的原因。

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

model = smf.ols("math ~ C(school) * C(gender)", data=df).fit()
aov = anova_lm(model, typ=2)
print(aov.round(4))

#### 📤 实际输出
学校（主效应）：F≈5.81, p≈0.003 → 显著。性别（主效应）：F≈1.71, p≈0.192 → 不显著。学校×性别（交互效应）：F≈1.16, p≈0.315 → 不显著。✅ 与课程一致 —— 学校确实造成真实差异，但这一差异在男女生之间并无明显不同（无交互效应）。

## 4. 交互效应可视化（两条线近似平行 = 无强交互效应）

In [ ]:
fig, ax = plt.subplots()
sns.pointplot(data=df, x="school", y="math", hue="gender", order=["A","B","C"], ax=ax, errorbar=None)
ax.set_title("按学校×性别分组的平均数学成绩")
plt.show()